# Protein-Protein Interaction Graph Preprocessing
Author: Cleverson Matiolli, Ph.D.

This notebook focuses on preprocessing of Protein-Protein Interaction (PPI) networks into torch graph format [PyTorch Geometric](http://www.pyg.org) for use in the Aid2GO Heterogeneous Graph Attention Network (Aid2GO-HAN).

**Key Steps:**
1. Import necessary libraries and load PPI data
2. Clean-up and format PPI dataset (handle missing data, convert dtypes and format)
3. Filter PPI dataset into annotated proteins (ground-truth GO terms) vs non-annotated
4. Extract PPI metadata (PPI detection methods, )
5. Construct a GO graph representation for use in the main Aid2GO model
6. Perform Exploratory Data Analysis (EDA) to analyze embedding quality


In [7]:
# Standard library imports
import math
import os
import pickle
import random
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Machine learning and data processing
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples
import hdbscan

# Bioinformatics and deep learning
from Bio import SeqIO
from obonet import read_obo
import networkx as nx
import torch
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, HeteroData
from torch_geometric.utils import degree, to_undirected

# Pre-trained models
from transformers import BertTokenizer, BertModel

# Configuration
pd.options.mode.copy_on_write = True

# Configuration of protein sequence length thresholds
min_length = 50  # still get small functional peptides
max_length = 4000  # way above the average protein lenght (500 residues)

### Convenience Functions

In [73]:
# PPI Dataset Preprocessing

def extract_uniprot_ids(field):
    """
    Extracts the UniProt ID from a given field.

    Parameters:
    field (str): A string containing UniProt IDs in the format "uniprotkb:ID".

    Returns:
    str or None: The extracted UniProt ID or None if not found.
    """
    if pd.isna(field):
        return None
    split_field = field.split("|")
    for uniprot_id in split_field:
        if uniprot_id.startswith("uniprotkb"):
            return uniprot_id.split(":")[1]
    return None


def extract_methods(field):
    """
    Extracts methods enclosed in parentheses from a given field.

    Parameters:
    field (str): A string containing methods in the format "method (detail)".

    Returns:
    str: Semi-colon separated methods or an empty string if not found.
    """
    if pd.isna(field):
        return None
    split_field = field.split("|")
    fragment_list = [
        re.search(r"\((.*?)\)", method).group(1)
        for method in split_field
        if re.search(r"\((.*?)\)", method)
    ]
    return ";".join(fragment_list) if fragment_list else ""


def extract_taxiIDs(field):
    """
    Extracts the first numeric ID from a given field.

    Parameters:
    field (str): A string containing numeric IDs.

    Returns:
    str or None: The extracted ID or None if not found.
    """
    if pd.isna(field):
        return None
    pattern = re.compile(r"\b\d+\b")
    match = pattern.search(field)
    return match.group(0) if match else None


def extract_ppi_type(field):
    """
    Extracts unique PPI types enclosed in parentheses from a given field.

    Parameters:
    field (str): A string containing PPI types in the format "type (detail)".

    Returns:
    str: Semi-colon separated unique PPI types or an empty string if not found.
    """
    if pd.isna(field):
        return None
    split_field = field.split("|")
    fragment_set = {
        re.search(r"\((.*?)\)", ppi_type).group(1)
        for ppi_type in split_field
        if re.search(r"\((.*?)\)", ppi_type)
    }
    return ";".join(fragment_set) if fragment_set else ""


def extract_confidence_score(field):
    """
    Extracts the confidence score from a given field.

    Parameters:
    field (str): A string containing confidence score in the format "label:score".

    Returns:
    str or None: The extracted confidence score or None if not found.
    """
    if pd.isna(field):
        return None
    parts = field.split(":")
    return parts[1] if len(parts) > 1 else None



In [74]:
# Protein sequence embedding

class ProteinDataset(Dataset):
    """
    A PyTorch Dataset class for reading and preprocessing protein sequences for embedding generation.

    Attributes:
        dataframe (pd.DataFrame): A dataframe with UniProt IDs, protein sequences (1-letter a. a. code)
        and their sequence lengths (column names: "uniprot_id", "sequence", "lenght").

    Methods:
        __len__: Returns the number of files in the dataset.
        __getitem__: Retrieves a sequence and its identifier by index, applying preprocessing.
    """

    def __init__(self, dataframe):
        """
        Initializes the ProteinDataset with a dataframe.

        Parameters:
            dataframe (pd.DataFrame): A dataframe with UniProt IDs, protein sequences (1-letter a. a. code)
        and their sequence lengths (column names: "uniprot_id", "sequence", "lenght").
        """
        self.dataframe = dataframe

    def __len__(self):
        """
        Provides the size of the dataset.

        Returns:
            int: The number of sequences in the dataframe.
        """
        return len(self.dataframe)

    def __getitem__(self, idx):
        """
        Retrieves the sequence and its identifier at the specified index after preprocessing.

        Parameters:
            idx (int): The index of the sequence to retrieve.

        Returns:
            tuple: A tuple containing the preprocessed sequence and its id.
        """
        row = self.dataframe.iloc[idx]
        seq = " ".join(row["sequence"].replace(r"[UZOB]", "X"))

        return seq, row["uniprot_id"]


class ProteinEmbeddingDataset(Dataset):
    """
    A PyTorch Dataset for loading protein embeddings stored as .pt (PyTorch tensor) files.

    Each tensor file contains the embedding of a protein, identified by a UniProt ID. This dataset
    facilitates the access to these embeddings for model training or inference, ensuring each
    embedding is loaded as needed, thus optimizing memory usage.

    Attributes:
        tensor_folder (str): The directory path where the embedding tensors (.pt files) are stored.
        uniprot_ids (list): A list of UniProt IDs corresponding to the proteins for which embeddings are available.

    Methods:
        __len__: Returns the total number of embeddings available in the dataset.
        __getitem__: Retrieves an embedding by its index, along with its corresponding UniProt ID.

    Parameters:
        tensor_folder (str): Path to the folder where tensor files, representing protein embeddings, are stored.
                             Each file should be named with the UniProt ID of the protein it represents.
    """

    def __init__(self, tensor_folder):
        """
        Initializes the dataset with the path to the directory containing the embeddings.

        Args:
            tensor_folder (string): Path to the folder where tensors are stored.
        """
        self.tensor_folder = tensor_folder
        self.uniprot_ids = [
            file.split(".")[0]
            for file in os.listdir(tensor_folder)
            if file.endswith(".pt")
        ]

    def __len__(self):
        """
        Returns the total number of protein embeddings available.

        Returns:
            int: The count of UniProt ID-based tensor files in the tensor folder.
        """
        return len(self.uniprot_ids)

    def __getitem__(self, idx):
        """
        Retrieves a specific protein embedding and its corresponding UniProt ID by index.

        This method ensures that the embedding tensor is loaded from disk only when accessed,
        optimizing memory usage, especially useful when dealing with large datasets.

        Args:
            idx (int): The index of the protein embedding to retrieve.

        Returns:
            dict: A dictionary containing the UniProt ID ('uniprot_id') and the embedding tensor ('tensor').
                  The tensor is expected to have a shape of (1, 1024).

        Raises:
            AssertionError: If the loaded tensor does not match the expected shape of (1, 1024).
        """
        uniprot_id = self.uniprot_ids[idx]
        tensor_path = os.path.join(self.tensor_folder, uniprot_id + ".pt")
        tensor = torch.load(tensor_path)

        # Asserting the tensor is in the correct shape ensures consistency and prevents potential errors in downstream processes.
        assert tensor.shape == (
            1,
            1024,
        ), f"Tensor shape mismatch: {tensor.shape} != (1, 1024)"

        return {"uniprot_id": uniprot_id, "tensor": tensor}


def embed_sequences(
    dataframe: pd.DataFrame, out_dir: Path, per_residue=False, batch_size=1
):
    """
    Processes protein sequences from a folder, generating embeddings using a pre-trained BERT model.

    Parameters:
        dataframe (pd.DataFrame): A dataframe with protein IDs, corresponding peptide sequences and their lengths.
        per_residue (bool): If True, saves per-residue embeddings; otherwise, saves per-protein embeddings.
        batch_size (int): The batch size for processing sequences.
    """

    print(f"Loading transformer and tokenizer...")
    tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
    model = BertModel.from_pretrained("Rostlab/prot_bert")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).half().eval()  # move model to gpu and set half-precision (float16)

    # Creates directories to save embeddings
    out_dir = Path(out_dir)  # ensures Path object
    out_dir.mkdir(parents=True, exist_ok=True)

    per_protein_path = out_dir / "per_protein"
    per_protein_path.mkdir(exist_ok=True, parents=True)

    per_residue_path = out_dir / "per_residue" if per_residue else None
    if per_residue_path:
        per_residue_path.mkdir(exist_ok=True, parents=True)

    # Reads fasta files in input folder
    dataset = ProteinDataset(dataframe)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    for seqs, ids in tqdm(loader, desc="Embedding sequences"):
        for seq, seq_id in zip(seqs, ids):

            # Tokenize and move input tensors to the same device as the model
            inputs = tokenizer(seq, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: tensor.to(device) for key, tensor in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
            protein_embeddings = outputs.last_hidden_state[
                :, : len(seq.replace(" ", "")) + 1
            ]

            # Detach and move to CPU before saving
            protein_embeddings = protein_embeddings.mean(dim=1).detach().cpu()
            torch.save(protein_embeddings, per_protein_path / f"{seq_id}.pt")

            if per_residue:
                residue_embeddings = outputs.last_hidden_state[
                    :, : len(seq.replace(" ", "")) + 1
                ]
                torch.save(
                    residue_embeddings.detach().cpu(),
                    per_residue_path / f"{seq_id}_res_embed.pt",
                )

    print(f"All proteins embedded successfully.")

In [75]:
# Create PPI graph dataset

def PPIDataset(
    source_embeddings_dir, dest_embeddings_dir, associations_df, ppi_edges, ppi_nodes, dataset="train"
):
    """
    Process protein embeddings, create a Protein-Protein Interaction (PPI) graph, and save the data.

    This function loads protein embeddings from .pt files, filters them based on the provided associations,
    constructs a PPI graph, computes degrees, plots the degree distribution, and saves the processed data.

    Args:
        data_folder (Path): Path to the main data directory.
        ppi_folder (Path): Path to the PPI-specific folder where outputs will be saved.
        associations_df (pd.DataFrame): DataFrame containing protein associations with at least a 'uniprot_id' column.
        ppi_edges (list): List of tuples representing edges in the PPI network.
        ppi_nodes (list): List of nodes in the PPI network.
        dataset (str, optional): Identifier for the dataset (e.g., 'train', 'test'). Defaults to "train".

    Returns:
        torch_geometric.data.Data: The processed PPI data containing node features and edge indices.
    """

    # Load embeddings directly from the folder
    source_embeddings_dir = Path(source_embeddings_dir)  # Ensures Path Object
    source_embedding_files = [
        f for f in os.listdir(source_embeddings_dir) if f.endswith(".pt")
    ]
    print(
        f"# embeddings in {source_embeddings_dir.as_posix()}: {len(source_embedding_files)}"
    )

    # Collect embeddings
    unique_proteins = set(associations_df["uniprot_id"].unique())
    protein_embed_dict = {}
    for file in tqdm(source_embedding_files, desc="Loading Protein Embeddings"):
        uniprot_id = file.split(".")[0]
        if uniprot_id in unique_proteins:
            tensor_path = source_embeddings_dir / file
            tensor = torch.load(tensor_path)
            assert tensor.shape == (
                1,
                1024,
            ), f"Tensor shape mismatch: {tensor.shape} != (1, 1024)"
            protein_embed_dict[uniprot_id] = tensor.numpy().flatten()

    # Save embeddings
    with open(dest_embeddings_dir / "protein_embeddings.pkl", "wb") as file:
        pickle.dump(protein_embed_dict, file)

    # Process embeddings into a feature matrix
    protein_ids = list(protein_embed_dict.keys())
    protein_feats = torch.stack(
        [
            torch.tensor(protein_embed_dict[pid], dtype=torch.float32)
            for pid in protein_ids
        ]
    )
    protein_ids_map = {pid: idx for idx, pid in enumerate(protein_ids)}

    print(f"# of protein IDs: {len(protein_ids)}")
    print(f"Protein features shape: {protein_feats.shape}")
    
    print(protein_ids_map)

    # Create edge index
    edges_to_tensor = []
    missing_proteins = set()
    for e in ppi_edges:
        if e[0] in protein_ids_map and e[1] in protein_ids_map:
            edges_to_tensor.append((protein_ids_map[e[0]], protein_ids_map[e[1]]))
        else:
            missing_proteins.update([e[0], e[1]])
            
    if missing_proteins:
        print(f"{len(missing_proteins)} missing proteins: {missing_proteins}")
        
    if edges_to_tensor:
        edge_index_ppi = torch.tensor(edges_to_tensor, dtype=torch.long).t().contiguous()
        print(f"Edge index shape: {edge_index_ppi.shape}")
    else:
        print("No valid edges found!")


    
    # edges_to_tensor = [
    #     [protein_ids_map[e[0]], protein_ids_map[e[1]]]
    #     for e in ppi_edges
    #     if e[0] in unique_proteins and e[1] in unique_proteins
    # ]
    # edge_index_ppi = torch.tensor(edges_to_tensor, dtype=torch.long).t().contiguous()
    # print(f"Edge index shape: {edge_index_ppi.shape}")

    # Convert to undirected graph and compute degrees
    edge_index_ppi = to_undirected(edge_index_ppi)
    degrees = degree(edge_index_ppi[0])

    # Plot degree distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(degrees.numpy(), bins=50, kde=True)
    plt.title("Distribution of Degrees")
    plt.xlabel("Degree")
    plt.ylabel("Frequency")
    plt.savefig(dest_embeddings_dir / f"degree_distribution_{dataset}.png")
    plt.close()

    # Create and save Data object
    ppi_data = Data(x=protein_feats, edge_index=edge_index_ppi)
    torch.save(ppi_data, dest_embeddings_dir / f"ppi_dataset_{dataset}.pt")

    return ppi_data, missing_proteins

In [76]:
def process_protein_sequences(
    data_folder, dataset="train", filter_annotated=True, min_length=50, max_length=4000
):
    """
    Load, parse, and filter CAFA5 sequences.

    Args:
    data_folder (Path): Path to the data folder.
    dataset (str): The dataset to process, either 'train' or 'test'. Default is 'train'.
    filter_annotated (bool): If True, filter out proteins with obsolete GO terms. Default is True.
    min_length (int): Minimum sequence length threshold. Default is 50.
    max_length (int): Maximum sequence length threshold. Default is 4000.

    Returns:
    pd.DataFrame: Filtered sequences DataFrame.
    """
    # Load sequences
    file_path = data_folder / f"cafa/{dataset}_sequences.fasta"
    sequences = [
        {
            "uniprot_id": record.id,
            "sequence": str(record.seq),
            "length": len(record.seq),
        }
        for record in SeqIO.parse(file_path, "fasta")
    ]

    # Create and filter by min and max sequence length
    sequences_df = pd.DataFrame(sequences)
    sequences_df = (
        sequences_df[
            (sequences_df["length"] <= max_length)
            & (sequences_df["length"] >= min_length)
        ]
        .sort_values(by="length", ascending=False)
        .reset_index(drop=True)
    )

    if filter_annotated:
        # Filter out proteins with obsolete GO terms
        associations_df = pd.read_csv(data_folder / "cafa/associations.tsv", sep="\t")
        valid_uniprot_ids = associations_df["uniprot_id"].unique()
        results_df = sequences_df[
            sequences_df["uniprot_id"].isin(valid_uniprot_ids)
        ].reset_index(drop=True)
    else:
        results_df = sequences_df

    # Plot histogram of protein sizes
    plt.figure(figsize=(4, 3))
    sns.histplot(results_df, x="length", binwidth=10).set(
        title="Protein Size Distribution (a.a. counts)"
    )
    plt.savefig(data_folder / f"cafa/cafa_protein_size_dist_{dataset}.png", dpi=300)
    plt.show()
    plt.close()

    # Save the data
    results_df.to_csv(data_folder / f"cafa/{dataset}sequences.csv", index=False)

    print(f"Sequences df dims: {results_df.shape}")
    print(results_df.head())

    return results_df

## Load GOA Annotations

In [ ]:
# Load ground-truth annotations (SwissProt)
associations_df = pd.read_csv(base_dir / "data/goa/goa_high_confidence.csv")
print(f"Annotations df shape: {associations_df.shape}")
associations_df.head()

In [ ]:
# Load Sequences
file_path = Path("/media/clever/aidbio/data/goa/uniprot_sprot.fasta")
sequences = [
    {
        "uniprot_id": record.id.split("|")[1],
        "sequence": str(record.seq),
        "length": len(record.seq),
    }
    for record in SeqIO.parse(file_path, "fasta")
]

# Filter by min and max sequence length and create df
sequences_df = pd.DataFrame(sequences)
sequences_df = (
    sequences_df[
        (sequences_df["length"] <= max_length) & (sequences_df["length"] >= min_length)
    ]
    .sort_values(by="length", ascending=False)
    .reset_index(drop=True)
)

# Save filtered sequences
sequences_df.to_csv(base_dir / "data/goa/uniprot_sprot.csv", index=False)

print(f"Sequences df shape: {sequences_df.shape}")
sequences_df.head()

# Plot histogram of protein sizes and save plot
save_path = base_dir / "eda/protein"
save_path.mkdir(parents=True, exist_ok=True)  # Ensure folder exists

plt.figure(figsize=(8, 6))
sns.histplot(sequences_df, x="length", binwidth=10, kde=True).set(
    title="Protein Size Distribution (a.a. counts)"
)
plt.tight_layout()
plt.savefig(save_path / "goa_high_confidence_protein_size_dist.png", dpi=300)
plt.show()
plt.close()

## Load and Format PPI data

In [ ]:
# Load human PPI data (source: Database of Interacting Proteins)
file_path = Path("/media/clever/aidbio/data/dip/human.txt")
ppi_df = pd.read_csv(
    file_path,
    delimiter=",",
    on_bad_lines="warn",
)

# Keep only relevant columns
ppi_df.drop(
    columns=[
        "Alt. ID interactor A",
        "Alt. ID interactor B",
        "Alias(es) interactor A",
        "Alias(es) interactor B",
        "Publication 1st author(s)",
        "Publication Identifier(s)",
        "Source database(s)",
        "Interaction identifier(s)",
        "Processing Status",
    ],
    inplace=True,
)


# Rename remaining columns
new_column_names = {
    "ID interactor A": "uniprot_id_A",
    "ID interactor B": "uniprot_id_B",
    "Interaction detection method(s)": "detection_method",
    "Taxid interactor A": "taxiID_A",
    "Taxid interactor B": "taxiID_B",
    "Interaction type(s)": "ppi_type",
    "Confidence value(s)": "confidence",
}

ppi_df.rename(columns=new_column_names, inplace=True)

# Reorder columns
ppi_df = ppi_df[
    [
        "uniprot_id_A",
        "uniprot_id_B",
        "ppi_type",
        "detection_method",
        "confidence",
        "taxiID_A",
        "taxiID_B",
    ]
]

# Format data
# Extract uniprot ids
ppi_df["uniprot_id_A"] = ppi_df["uniprot_id_A"].apply(extract_uniprot_ids)
ppi_df["uniprot_id_B"] = ppi_df["uniprot_id_B"].apply(extract_uniprot_ids)

# Extract methods
ppi_df["detection_method"] = ppi_df["detection_method"].apply(extract_methods)

# Extract taxIDs
ppi_df["taxiID_A"] = ppi_df["taxiID_A"].apply(extract_taxiIDs)
ppi_df["taxiID_B"] = ppi_df["taxiID_B"].apply(extract_taxiIDs)

# Extract PPI type
ppi_df["ppi_type"] = ppi_df["ppi_type"].apply(extract_ppi_type)

# Extract confidence score
ppi_df["confidence"] = ppi_df["confidence"].apply(extract_confidence_score)

# Remove PPI pairs of diferent organisms
ppi_df = ppi_df[ppi_df["taxiID_A"] == ppi_df["taxiID_B"]]
ppi_df.reset_index(drop=True, inplace=True)

print(ppi_df.info())
ppi_df.head()

## Split PPI Dataset into Annotated and Non-Annotated

### Create Dataset of PPIs without Annotations (Model Evaluations)

In [ ]:
# Filter associations df by identifiers present in sequences df (filtered by size)

associations_df = associations_df[
    associations_df["uniprot_id"].isin(sequences_df["uniprot_id"].unique())
]

associations_df

In [ ]:
# Extract unique protein ids
unique_associations_protein_ids = associations_df["uniprot_id"].unique()

# Extract entries without annotation
ppi_no_annot_df = ppi_df[
    ~(ppi_df["uniprot_id_A"].isin(unique_associations_protein_ids))
    & (ppi_df["uniprot_id_B"].isin(unique_associations_protein_ids))
]
ppi_no_annot_df = ppi_no_annot_df[
    (ppi_no_annot_df["uniprot_id_A"] != ppi_no_annot_df["uniprot_id_B"])
]  # Drop self-interactions (A = B)

ppi_no_annot_df.dropna(inplace=True) # Drop unpaired ids
ppi_no_annot_df.reset_index(drop=True, inplace=True)

# Create directory and save ppi data
save_path = base_dir / "data/ppi"
save_path.mkdir(parents=True, exist_ok=True)
ppi_no_annot_df.to_csv(save_path / "ppi_unknown.csv", index=False)

print(ppi_no_annot_df.info())
ppi_no_annot_df.head()

In [ ]:
# PPI counts (unique)

# Extract interactions
ppi_pairs = list(
    tuple(sorted(x))
    for x in ppi_no_annot_df[["uniprot_id_A", "uniprot_id_B"]].to_numpy()
)

# Extract unique interactions
ppi_unique_pairs = list(
    set(
        tuple(sorted(x))
        for x in ppi_no_annot_df[["uniprot_id_A", "uniprot_id_B"]].to_numpy()
    )
)

# Extract unique uniprot ids in interactions
ppi_unique_ids = pd.concat(
    [ppi_no_annot_df["uniprot_id_A"], ppi_no_annot_df["uniprot_id_B"]]
).unique()

print(f"# of unique UniProt IDs: {ppi_unique_ids.shape[0]}")
print(f"# of PPIs: {len(ppi_pairs)}")
print(f"# of unique PPIs: {len(ppi_unique_pairs)}")

### Create Dataset of PPI with Annotations (for training and testing)

In [ ]:
# Filter out proteins without annotations
ppi_df = ppi_df[
    (ppi_df["uniprot_id_A"].isin(unique_associations_protein_ids))
    & (ppi_df["uniprot_id_B"].isin(unique_associations_protein_ids))
]

# Remove proteins without embeddings (large proteins)
unique_sequences_protein_ids = sequences_df["uniprot_id"].unique()
ppi_df = ppi_df[
    (ppi_df["uniprot_id_A"].isin(unique_sequences_protein_ids))
    & (ppi_df["uniprot_id_B"].isin(unique_sequences_protein_ids))
]
ppi_df = ppi_df[
    (ppi_df["uniprot_id_A"] != ppi_df["uniprot_id_B"])
]  # Drop self-interactions

ppi_df.dropna(inplace=True)  # Drop unpaired ids
ppi_df.reset_index(drop=True, inplace=True)
ppi_df.to_csv(save_path / "ppi.csv", index=False)

print(f"PPI df shape: {ppi_df.shape}")
print(ppi_df.head().to_markdown())
ppi_df.describe()

## Extract PPI metadata

In [ ]:
# Check PPI types in the dataset

# Get unique lists of ppi types
ppi_types = ppi_df["ppi_type"].unique()

# Get unique PPI types from lists of PPI types
unique_ppi_types = set(
    ppi_type.strip() for sublist in ppi_types for ppi_type in sublist.split(";")
)
unique_ppi_types = sorted(unique_ppi_types)  # convert to list

# Save list
with open(base_dir / "data/ppi/ppi_types.txt", "w") as f:
    f.write(str(unique_ppi_types))

print(f"# of unique PPI types: {len(unique_ppi_types)}")
print(unique_ppi_types)

In [ ]:
# Check PPI detection methods in the dataset

# Get unique lists of detection methods
methods = ppi_df["detection_method"].unique()

# Get unique PPI types from lists of PPI types
unique_methods = set(
    method.strip() for sublist in methods for method in sublist.split(";")
)
unique_methods = sorted(unique_methods)  # convert to list

# Save list
with open(base_dir / "data/ppi/ppi_methods.txt", "w") as f:
    f.write(str(unique_methods))

print(f"# of unique PPI detection methods: {len(unique_methods)}")
print(unique_methods[:5])

## Extract nodes and edges to build the PPI graph

In [ ]:
# PPI counts (unique)

# Extract interactions
ppi_pairs = list(
    tuple(sorted(x)) for x in ppi_df[["uniprot_id_A", "uniprot_id_B"]].to_numpy()
)

# Extract unique interactions
ppi_unique_pairs = list(
    set(tuple(sorted(x)) for x in ppi_df[["uniprot_id_A", "uniprot_id_B"]].to_numpy())
)

# Extract unique uniprot ids in interactions
ppi_unique_ids = pd.concat([ppi_df["uniprot_id_A"], ppi_df["uniprot_id_B"]]).unique()

print(f"# of unique UniProt IDs: {ppi_unique_ids.shape[0]}")
print(f"# of PPIs: {len(ppi_pairs)}")
print(f"# of unique PPIs: {len(ppi_unique_pairs)}")

## Extract Largest Connected Component from PPI graph (ensures connectivity)

In [ ]:
ppi_graph = nx.Graph() # Create undirected graph instance
ppi_graph.add_edges_from(ppi_unique_pairs)  # Add edges
cc = list(nx.connected_components(ppi_graph)) # Get connected components
lcc = max(cc, key=len) # Get the largest connected component (LCC)
ppi_graph_lcc = ppi_graph.subgraph(lcc).copy()  # Extract subgraph

# Extract nodes and edges for graph dataset
ppi_nodes = ppi_graph_lcc.nodes()
ppi_edges = list(ppi_graph_lcc.edges())

# print-out basic graph attributes
print(f"# of connected components: {len(cc)}")
print("\nFull PPI graph attributes:")
print(f"# of nodes: {ppi_graph.number_of_nodes()}")
print(f"# of edges: {ppi_graph.number_of_edges()}")
print(f"Is connected?: {nx.is_connected(ppi_graph_lcc)}")

print("\nExtracted PPI graph attributes:")
print(f"# of nodes (LCC): {len(ppi_nodes)}")
print(f"# of edges (LCC): {len(ppi_edges)}")
print(f"Is connected? (LCC): {nx.is_connected(ppi_graph_lcc)}")

## Filter-out disconnected components

In [ ]:
# Filter protein-go associations with LCC nodes

# Keep full protein-GO associations for plotting
associations_full = associations_df.copy()

# Remove associations with large proteins
associations_df = associations_df[
    associations_df["uniprot_id"].isin(list(ppi_graph_lcc.nodes()))
].copy()

# Save filtered df
file_path = base_dir / "data/ppi/associations_human.tsv"
associations_df.to_csv(file_path, sep="\t", index=False)


print(f"Filtered Protein-GO associations dims: {associations_df.shape}")
print(f"# of unique UniProt IDs: {len(associations_df['uniprot_id'].unique())}")
associations_df.head()

In [ ]:
associations_full

## Create PPI graph

In [ ]:
source_embeddings_dir = base_dir / "data/protein/embeddings/per_protein/"
dest_embeddings_dir = base_dir / "data/ppi"
ppi_data_human, missing_proteins = PPIDataset(
    source_embeddings_dir=source_embeddings_dir,
    dest_embeddings_dir= base_dir / "data/ppi/",
    associations_df=associations_df,
    ppi_edges=ppi_edges,
    ppi_nodes=ppi_nodes,
    dataset="train"
)

# Save missing proteins list
np.save("missing_proteins.npy", np.array(list(missing_proteins)), allow_pickle=True)

print(f"Average degree: {degree(ppi_data_human.edge_index[0]).mean():.2f}")
ppi_data_human